# Modul 11: Metriken, Suche und erklärbare Merkmale | Lösungen

## Überblick

Sie wählen Metriken passend zu Fehlerfolgen, führen leakage-sichere Kreuzvalidierung durch, optimieren kleine Suchräume und konstruieren oder wählen Merkmale innerhalb von Pipelines. Zum Abschluss erklären Sie Modellentscheidungen mit Koeffizienten und Permutationswichtigkeit.

**Zugehörige Vorlesungen**

- **Metriken und Suche**
- **Merkmale erklären**

## Lernziele

Nach der Bearbeitung können Sie:

- Klassifikations- und Regressionsmetriken einschließlich ROC- und Precision-Recall-Kurven fachlich auswählen.
- Kreuzvalidierung und Hyperparametersuche mit passenden Splittern leakage-sicher durchführen.
- Merkmalskonstruktion, Auswahl, Reduktion und Modellinterpretation in reproduzierbaren Pipelines verbinden.

## Geprüfte Fähigkeiten

- Schwellenanalyse, ROC-AUC, Average Precision, MAE und RMSE
- StratifiedKFold, GroupKFold, cross_validate, GridSearchCV und RandomizedSearchCV
- PolynomialFeatures, SelectKBest, PCA, Koeffizienten und permutation_importance

## Hinweise zur Bearbeitung

Dieses Lösungsnotebook enthält dieselben Aufgaben wie das Übungsnotebook sowie vollständige, ausführlich kommentierte Musterlösungen. Bearbeiten Sie nach Möglichkeit zuerst das Übungsnotebook und nutzen Sie dieses Dokument anschließend zur Kontrolle und Vertiefung.

- **Erwarteter Schwierigkeitsgrad:** fortgeschritten
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Die Setup-Zelle lädt den kleinen Brustkrebs-Datensatz aus scikit-learn und erzeugt zusätzlich Gruppenkennungen sowie einen kleinen nichtlinearen Regressionsdatensatz. Alle Daten bleiben lokal verfügbar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

krebs = load_breast_cancer(as_frame=True)
X_klassifikation = krebs.data.copy()
y_klassifikation = krebs.target.copy()

X_train, X_test, y_train, y_test = train_test_split(
    X_klassifikation,
    y_klassifikation,
    test_size=0.25,
    stratify=y_klassifikation,
    random_state=RANDOM_SEED,
)

# Künstliche Gruppen simulieren mehrere Messungen aus derselben Quelle.
gruppen = np.repeat(np.arange(0, int(np.ceil(len(X_klassifikation) / 4))), 4)[: len(X_klassifikation)]

# Kleiner nichtlinearer Regressionsdatensatz für Merkmalskonstruktion.
x_reg = np.linspace(-3.0, 3.0, 180)
y_reg = 2.0 + 1.2 * x_reg - 0.9 * x_reg**2 + rng.normal(0.0, 0.8, size=x_reg.size)
X_reg = x_reg.reshape(-1, 1)

print("Klassifikationsdaten:", X_klassifikation.shape)
print("Regressionsdaten:", X_reg.shape)

### Aufgabe 1: Schwellenwerte und Fehlerfolgen untersuchen

Trainieren Sie eine skalierte logistische Regression. Berechnen Sie für die Schwellenwerte `0.30`, `0.50` und `0.70` jeweils Accuracy, Precision, Recall und F1. Stellen Sie die Ergebnisse tabellarisch dar und bestimmen Sie den Schwellenwert mit dem höchsten Recall.

Begründen Sie anschließend, warum ein hoher Recall in einer medizinischen Screening-Situation wichtig sein kann und welche Gegenleistung dafür häufig entsteht.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# Die Skalierung wird innerhalb der Pipeline ausschließlich aus den Trainingsdaten gelernt.
# Das verhindert, dass Informationen aus dem Testdatensatz in die Vorverarbeitung gelangen.
modell = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=RANDOM_SEED),
)
modell.fit(X_train, y_train)

# predict_proba() liefert für jede Klasse eine Wahrscheinlichkeit.
# Spalte 1 gehört zur positiven Klasse des Datensatzes.
positive_wahrscheinlichkeiten = modell.predict_proba(X_test)[:, 1]

zeilen = []
for schwelle in [0.30, 0.50, 0.70]:
    vorhersage = (positive_wahrscheinlichkeiten >= schwelle).astype(int)
    zeilen.append(
        {
            "Schwelle": schwelle,
            "Accuracy": accuracy_score(y_test, vorhersage),
            "Precision": precision_score(y_test, vorhersage, zero_division=0),
            "Recall": recall_score(y_test, vorhersage, zero_division=0),
            "F1": f1_score(y_test, vorhersage, zero_division=0),
        }
    )

schwellenvergleich = pd.DataFrame(zeilen)
display(schwellenvergleich.round(3))

beste_recall_zeile = schwellenvergleich.loc[schwellenvergleich["Recall"].idxmax()]
print("Höchster Recall bei Schwelle:", beste_recall_zeile["Schwelle"])

> **Musterantwort und Interpretation**
>
> Bei einem Screening sollen möglichst wenige tatsächlich positive Fälle übersehen werden. Deshalb ist hoher Recall oft wichtig. Eine niedrigere Schwelle stuft jedoch mehr Fälle als positiv ein und erzeugt typischerweise zusätzliche Fehlalarme, wodurch die Precision sinken kann. Die endgültige Schwelle muss medizinische Folgen, Nachuntersuchungskosten und Belastungen für Betroffene berücksichtigen.

### Aufgabe 2: ROC- und Precision-Recall-Kurven vergleichen

Berechnen und visualisieren Sie für das trainierte Modell eine ROC-Kurve und eine Precision-Recall-Kurve. Geben Sie ROC-AUC und Average Precision aus. Markieren Sie in beiden Diagrammen die Leistung einer zufälligen beziehungsweise konstanten Referenz.

In [ ]:
from sklearn.metrics import (
    RocCurveDisplay,
    PrecisionRecallDisplay,
    roc_auc_score,
    average_precision_score,
)

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# ROC-AUC bewertet die Rangordnung positiver und negativer Fälle über alle Schwellen.
roc_auc = roc_auc_score(y_test, positive_wahrscheinlichkeiten)
# Average Precision fasst die Precision-Recall-Kurve zusammen und ist bei seltenen Positiven oft besonders informativ.
average_precision = average_precision_score(y_test, positive_wahrscheinlichkeiten)

print(f"ROC-AUC: {roc_auc:.3f}")
print(f"Average Precision: {average_precision:.3f}")

RocCurveDisplay.from_predictions(y_test, positive_wahrscheinlichkeiten)
plt.plot([0, 1], [0, 1], linestyle=":", label="Zufällige Rangfolge")
plt.title("ROC-Kurve auf dem Testdatensatz")
plt.legend()
plt.show()

PrecisionRecallDisplay.from_predictions(y_test, positive_wahrscheinlichkeiten)
positiver_anteil = float(np.mean(y_test))
plt.axhline(positiver_anteil, linestyle=":", label="Konstante Referenz")
plt.title("Precision-Recall-Kurve auf dem Testdatensatz")
plt.legend()
plt.show()

> **Musterantwort und Interpretation**
>
> Bei stark unausgeglichenen Klassen kann eine ROC-Kurve trotz vieler falsch positiver Ergebnisse günstig wirken, weil die große negative Klasse die False-Positive-Rate klein hält. Die Precision-Recall-Kurve konzentriert sich stärker auf die Qualität der positiven Vorhersagen und ist deshalb häufig geeigneter, wenn die positive Klasse selten und fachlich entscheidend ist.

### Aufgabe 3: Kreuzvalidierungsstrategien leakage-sicher prüfen

Vergleichen Sie für dieselbe Pipeline drei Bewertungsvarianten:

1. `StratifiedKFold` mit fünf Folds,
2. `GroupKFold` mit fünf Folds und den vorgegebenen Gruppen,
3. einen absichtlich ungeeigneten Split, bei dem die vollständigen Daten vor der Kreuzvalidierung skaliert werden.

Berechnen Sie jeweils den mittleren F1-Wert. Erklären Sie, warum Variante 3 methodisch falsch ist, auch wenn ihre Kennzahl nicht immer deutlich höher ausfällt.

In [ ]:
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_score

# ============================================================
# MUSTERLÖSUNG
# ============================================================

stratifizierte_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
gruppen_cv = GroupKFold(n_splits=5)

saubere_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=RANDOM_SEED),
)

f1_stratifiziert = cross_val_score(
    saubere_pipeline,
    X_klassifikation,
    y_klassifikation,
    cv=stratifizierte_cv,
    scoring="f1",
)

f1_gruppen = cross_val_score(
    saubere_pipeline,
    X_klassifikation,
    y_klassifikation,
    groups=gruppen,
    cv=gruppen_cv,
    scoring="f1",
)

# Diese Variante ist absichtlich methodisch falsch:
# Der Scaler sieht Mittelwerte und Standardabweichungen aller späteren Validierungsfolds.
# Das Beispiel dient nur dazu, Leakage erkennen zu lernen.
X_vorab_skaliert = StandardScaler().fit_transform(X_klassifikation)
f1_leakage = cross_val_score(
    LogisticRegression(max_iter=2000, random_state=RANDOM_SEED),
    X_vorab_skaliert,
    y_klassifikation,
    cv=stratifizierte_cv,
    scoring="f1",
)

cv_vergleich = pd.DataFrame(
    {
        "Verfahren": ["StratifiedKFold", "GroupKFold", "Vorab skalierte Daten"],
        "F1_Mittel": [f1_stratifiziert.mean(), f1_gruppen.mean(), f1_leakage.mean()],
        "F1_Std": [f1_stratifiziert.std(), f1_gruppen.std(), f1_leakage.std()],
    }
)
display(cv_vergleich.round(3))

> **Musterantwort und Interpretation**
>
> Jeder Validierungsfold soll eine bisher ungesehene Datenmenge simulieren. Wird die Skalierung vorab auf allen Daten angepasst, beeinflussen Verteilungsinformationen aus den Validierungsfolds die Trainingsdarstellung. Eine Pipeline passt den Scaler in jedem Fold nur auf dem jeweiligen Trainingsanteil an. Methodische Korrektheit hängt nicht davon ab, ob der Leakage-Effekt in einem einzelnen Datensatz groß sichtbar wird.

### Aufgabe 4: Kleine Hyperparametersuche dokumentieren

Führen Sie eine `GridSearchCV` für eine Pipeline aus Standardisierung und logistischer Regression durch. Prüfen Sie ausschließlich die Werte `C = [0.01, 0.1, 1.0, 10.0]`. Nutzen Sie stratifizierte fünfteilige Kreuzvalidierung und F1 als Zielmetrik.

Erstellen Sie aus `cv_results_` eine kompakte Tabelle mit Parameterwert, mittlerem F1-Wert, Standardabweichung und Rang. Bewerten Sie das beste Modell einmalig auf dem unberührten Testdatensatz.

In [ ]:
from sklearn.model_selection import GridSearchCV

# ============================================================
# MUSTERLÖSUNG
# ============================================================

such_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=RANDOM_SEED),
)

suche = GridSearchCV(
    estimator=such_pipeline,
    param_grid={"logisticregression__C": [0.01, 0.1, 1.0, 10.0]},
    scoring="f1",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED),
    n_jobs=-1,
    return_train_score=False,
)
suche.fit(X_train, y_train)

suchbericht = pd.DataFrame(suche.cv_results_)[
    ["param_logisticregression__C", "mean_test_score", "std_test_score", "rank_test_score"]
].rename(
    columns={
        "param_logisticregression__C": "C",
        "mean_test_score": "F1_Mittel",
        "std_test_score": "F1_Std",
        "rank_test_score": "Rang",
    }
).sort_values("Rang")

display(suchbericht.round(4))
print("Beste Parameter:", suche.best_params_)
print("Test-F1:", round(f1_score(y_test, suche.predict(X_test)), 3))

> **Musterantwort und Interpretation**
>
> Der Testdatensatz soll eine einmalige, möglichst unverzerrte Schätzung der Leistung nach allen Modellentscheidungen liefern. Wird C anhand des Tests gewählt, wird der Test indirekt Teil des Trainingsprozesses und die berichtete Leistung ist optimistisch.

### Aufgabe 5: Nichtlineare Merkmale und Regularisierung verbinden

Teilen Sie den Regressionsdatensatz reproduzierbar auf. Vergleichen Sie:

- eine lineare Ridge-Regression nur mit `x`,
- eine Pipeline aus `PolynomialFeatures(degree=2)`, Standardisierung und Ridge.

Berechnen Sie RMSE und R² auf dem Testdatensatz und zeichnen Sie die Vorhersagekurven über einem geordneten Gitter. Diskutieren Sie, weshalb der Polynomgrad innerhalb der Pipeline erzeugt werden sollte.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import PolynomialFeatures

# ============================================================
# MUSTERLÖSUNG
# ============================================================

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg,
    y_reg,
    test_size=0.25,
    random_state=RANDOM_SEED,
)

lineares_modell = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
polynom_modell = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    StandardScaler(),
    Ridge(alpha=1.0),
)

modelle = {"Linear": lineares_modell, "Polynomgrad 2": polynom_modell}
reg_zeilen = []
for name, aktuelles_modell in modelle.items():
    aktuelles_modell.fit(Xr_train, yr_train)
    prognose = aktuelles_modell.predict(Xr_test)
    reg_zeilen.append(
        {
            "Modell": name,
            "RMSE": mean_squared_error(yr_test, prognose) ** 0.5,
            "R2": r2_score(yr_test, prognose),
        }
    )

display(pd.DataFrame(reg_zeilen).round(3))

x_gitter = np.linspace(X_reg.min(), X_reg.max(), 300).reshape(-1, 1)
plt.scatter(Xr_test[:, 0], yr_test, alpha=0.65, label="Testdaten")
for name, aktuelles_modell in modelle.items():
    plt.plot(x_gitter[:, 0], aktuelles_modell.predict(x_gitter), label=name)
plt.xlabel("x")
plt.ylabel("Zielwert")
plt.title("Lineare und quadratische Merkmalsdarstellung")
plt.legend()
plt.show()

> **Musterantwort und Interpretation**
>
> Mit hohem Polynomgrad wächst die Zahl der Merkmale und damit die Flexibilität stark. Das Modell kann dann Rauschen und zufällige Einzelheiten der Trainingsdaten nachbilden. Regularisierung und Kreuzvalidierung begrenzen dieses Überanpassungsrisiko, ersetzen aber keine fachlich sinnvolle Komplexitätswahl.

### Aufgabe 6: Integrationsaufgabe: Auswahl und Interpretation in einer Pipeline

Bauen Sie eine Pipeline aus Standardisierung, `SelectKBest(f_classif, k=10)` und logistischer Regression. Trainieren Sie sie auf den Trainingsdaten und bewerten Sie F1 auf dem Testdatensatz.

Ermitteln Sie anschließend:

1. die zehn ausgewählten Originalmerkmale,
2. die standardisierten Koeffizienten des Klassifikators,
3. die fünf wichtigsten Testmerkmale nach Permutationswichtigkeit.

Erklären Sie, warum Koeffizienten und Permutationswichtigkeit nicht zwingend dieselbe Rangfolge liefern.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.inspection import permutation_importance

# ============================================================
# MUSTERLÖSUNG
# ============================================================

erklaer_pipeline = make_pipeline(
    StandardScaler(),
    SelectKBest(score_func=f_classif, k=10),
    LogisticRegression(max_iter=2000, random_state=RANDOM_SEED),
)
erklaer_pipeline.fit(X_train, y_train)

test_vorhersage = erklaer_pipeline.predict(X_test)
print("Test-F1:", round(f1_score(y_test, test_vorhersage), 3))

# get_support() bezieht sich auf die Spaltenreihenfolge vor der Auswahl.
auswahl_schritt = erklaer_pipeline.named_steps["selectkbest"]
ausgewaehlt = X_train.columns[auswahl_schritt.get_support()]

# Die Koeffizienten beziehen sich nur auf die zehn nach SelectKBest verbleibenden Merkmale.
koeffizienten = erklaer_pipeline.named_steps["logisticregression"].coef_[0]
koeffizienten_tabelle = pd.DataFrame(
    {
        "Merkmal": ausgewaehlt,
        "Koeffizient": koeffizienten,
        "Absoluter_Koeffizient": np.abs(koeffizienten),
    }
).sort_values("Absoluter_Koeffizient", ascending=False)
print("Ausgewählte Merkmale und Koeffizienten:")
display(koeffizienten_tabelle)

# Permutationswichtigkeit wird auf der vollständigen Pipeline berechnet.
# Deshalb erhalten wir eine Wichtigkeit für jedes ursprüngliche Eingabemerkmal.
permutation = permutation_importance(
    erklaer_pipeline,
    X_test,
    y_test,
    scoring="f1",
    n_repeats=15,
    random_state=RANDOM_SEED,
)
permutation_tabelle = pd.DataFrame(
    {
        "Merkmal": X_test.columns,
        "F1_Abnahme": permutation.importances_mean,
        "Std": permutation.importances_std,
    }
).sort_values("F1_Abnahme", ascending=False)
print("Fünf wichtigste Merkmale nach Permutation:")
display(permutation_tabelle.head(5).round(4))

> **Musterantwort und Interpretation**
>
> Ein Koeffizient beschreibt den Einfluss eines standardisierten Merkmals innerhalb des angepassten linearen Modells, während alle anderen Modellmerkmale konstant gedacht werden. Permutationswichtigkeit misst dagegen den tatsächlichen Leistungsabfall, wenn ein Merkmal im Testdatensatz zerstört wird. Korrelationen, Merkmalsauswahl und redundante Informationen können den Leistungsabfall abschwächen, obwohl ein Koeffizient groß ist. Beide Verfahren liefern modellbezogene Hinweise und keine automatische kausale Erklärung.

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?